# What NCHW and NHWC actually mean

Both describe memory layout for 4-D tensors used in CNNs.

Assume a tensor with:
- $N$: batch size
- $C$: channels(features map)
- $W$: width
- $H$: height

## NCHW

$$
[N, C, H, W]
$$

Memory order (fastest → slowest varying):

$$
W \rightarrow H \rightarrow C \rightarrow N
$$

## NHWC

$$
[N, H, W, C]
$$

Memory order:

$$
C \rightarrow W \rightarrow H \rightarrow N
$$

#### $\rightarrow$ The difference is where channels live in memory.

# Why this matters (not just naming)

This choice directly affects:
- Cache locality
- Vectorization (SIMD)
- GPU kernel design
- Convolution lowering (im2col / GEMM)
- Framework & hardware performance

#### Same math, very different performance characteristics.

# NCHW: why it exists and when it's used

## Key property

Channels are contiguous in a block per feature map.

Channel 0: H×W

Channel 1: H×W

...

## Advantages

- Natural for GEMM-based convolutions
- Efficient when:
    - Channels are large
    - Compute diminates memory bandwidth
- Historically aligned with cuDNN, CUDA kernels

# Typical use cases

- NVIDIA GPUs
- PyTorch default
- cuDNN / training workloads
- Large CNNs (ResNet, EfficientNet)

## Why GPUs liked it (historically)

- Convolution → im2col → matrix multiply
- Matrix layout maps cleanly when channels are grouped

#### $\rightarrow$ NCHW optimized compute throughput

# NHWC: why it exists and when it's used

## Key property

Channels are adjacent per pixel.

Pixel (h, w): [c0, c1, c2, ...]

## Advantages

- Better memory coalescing for:
    - Small channel counts
    - Depthwise & pointwise convs
- Friendly to SIMD on CPUs
- lower cache misses

## Typical Use Cases

- TensorFlow default
- Mobile / Edge devices
- ARM CPUs
- TPU
- Depthwise separable conv (MobileNet)

## Why NHWC wins on mobile

- Pixel-wise ops (ReLU, BN, DWConv) dominate
- Channel-last aligns with vector registers
- Less transpose overhead

#### $\rightarrow$ NHWC optimized memory bandwidth & cache efficiency

# Framework & Hardware mapping

![](image1.png)

# Rule of Thumb

- Training / NVIDIA GPU / Large CNN $\rightarrow$ NCHW
- Inference / mobile / edge / depthwise-heavy $\rightarrow$ NHWC
- CPU-bound pipelines $\rightarrow$ NHWC
- Mixed ops $\rightarrow$ pick one and stick to it end-to-end